# Level 3 — Task 1: Predictive Analytics & Machine Learning
**Internship:** Codveda Technology — Business Analytics  
**Objective:** Use machine learning to predict business outcomes — regression for house price forecasting, classification for churn prediction, and clustering for market segmentation.  
**Datasets:** churn_cleaned.csv · house_Prediction_Data_Set.csv · iris_cleaned.csv

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, silhouette_score)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
np.random.seed(42)
print('Libraries loaded ✓')

---
## 1. Load Datasets

In [ ]:
DATA_DIR = '../data'
churn = pd.read_csv(os.path.join(DATA_DIR, 'churn_cleaned.csv'))
BOSTON_COLS = ['CRIM','ZN','INDUS','CHAS','NOX','RM','AGE','DIS','RAD','TAX','PTRATIO','B','LSTAT','MEDV']
house = pd.read_csv(os.path.join(DATA_DIR, '4__house_Prediction_Data_Set.csv'), header=None, sep=r'\s+', names=BOSTON_COLS)
iris  = pd.read_csv(os.path.join(DATA_DIR, 'iris_cleaned.csv'))
print(f'Churn: {churn.shape} | House: {house.shape} | Iris: {iris.shape}')
print('\nBoston Housing — Feature Guide:')
display(pd.DataFrame({'Feature':BOSTON_COLS,'Description':['Crime rate','Residential land zone','Non-retail business','Charles River (0/1)','NOx concentration','Avg rooms/dwelling','Pre-1940 units %','Distance to employment','Highway access index','Property tax rate','Pupil-teacher ratio','B statistic','% lower status population','Median home value ($000s) TARGET']}))

---
## 2. Regression — House Price Prediction
**Business Goal:** Predict median home values to support real estate pricing and investment decisions.  
Models compared: Linear Regression, Ridge, Lasso, Random Forest, Gradient Boosting.

In [ ]:
X_house = house.drop('MEDV', axis=1)
y_house = house['MEDV']
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_house, y_house, test_size=0.2, random_state=42)
scaler_h = StandardScaler()
X_train_h_sc = scaler_h.fit_transform(X_train_h)
X_test_h_sc  = scaler_h.transform(X_test_h)
print(f'Train: {X_train_h.shape} | Test: {X_test_h.shape}')

In [ ]:
def evaluate_regressor(model, X_tr, y_tr, X_te, y_te, name):
    """Reusable: trains and evaluates a regression model. Returns metrics dict."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)
    r2   = r2_score(y_te, y_pred)
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()
    print(f'  {name:<28} RMSE={rmse:.3f}  MAE={mae:.3f}  R²={r2:.4f}  CV-R²={cv:.4f}')
    return {'model':model,'name':name,'y_pred':y_pred,'rmse':rmse,'mae':mae,'r2':r2,'cv_r2':cv}

print('-- Regression Model Comparison --')
reg_models = [
    (LinearRegression(), X_train_h_sc, X_test_h_sc, 'Linear Regression'),
    (Ridge(alpha=1.0), X_train_h_sc, X_test_h_sc, 'Ridge (a=1.0)'),
    (Lasso(alpha=0.1), X_train_h_sc, X_test_h_sc, 'Lasso (a=0.1)'),
    (RandomForestRegressor(n_estimators=100, random_state=42), X_train_h, X_test_h, 'Random Forest'),
    (GradientBoostingRegressor(n_estimators=100, random_state=42), X_train_h, X_test_h, 'Gradient Boosting'),
]
reg_results = []
for model, X_tr, X_te, name in reg_models:
    reg_results.append(evaluate_regressor(model, X_tr, y_train_h, X_te, y_test_h, name))

In [ ]:
best_reg = max(reg_results, key=lambda x: x['r2'])
print(f'\nBest model: {best_reg["name"]}  R²={best_reg["r2"]:.4f}')
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
# Actual vs predicted
axes[0].scatter(y_test_h, best_reg['y_pred'], alpha=0.6, s=30, color='#3498db', edgecolor='white')
mn,mx = y_test_h.min(), y_test_h.max()
axes[0].plot([mn,mx],[mn,mx],'r--',linewidth=2,label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($000s)', fontweight='bold')
axes[0].set_ylabel('Predicted Price ($000s)', fontweight='bold')
axes[0].set_title(f'Actual vs Predicted\n{best_reg["name"]}  R²={best_reg["r2"]:.4f}', fontweight='bold')
axes[0].legend()
# Model comparison
names=[r['name'] for r in reg_results]; r2s=[r['r2'] for r in reg_results]
bars=axes[1].barh(names, r2s, color='#3498db', edgecolor='white')
axes[1].set_xlabel('R² Score', fontweight='bold'); axes[1].set_title('Model Comparison -- R²', fontweight='bold')
axes[1].set_xlim(0,1.05)
for bar,val in zip(bars,r2s):
    axes[1].text(val+0.01, bar.get_y()+bar.get_height()/2, f'{val:.4f}', va='center', fontsize=9, fontweight='bold')
# Feature importance
rf_reg = next(r['model'] for r in reg_results if r['name']=='Random Forest')
feat_imp = pd.Series(rf_reg.feature_importances_, index=X_house.columns).sort_values(ascending=True).tail(10)
feat_imp.plot(kind='barh', ax=axes[2], color='#2ecc71', edgecolor='white')
axes[2].set_title('Feature Importance\n(Random Forest)', fontweight='bold')
plt.suptitle('Regression -- House Price Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('regression_house_price.png', bbox_inches='tight')
plt.show()

In [ ]:
residuals = y_test_h.values - best_reg['y_pred']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(best_reg['y_pred'], residuals, alpha=0.5, s=25, color='#9b59b6')
axes[0].axhline(0, color='#e74c3c', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Values', fontweight='bold'); axes[0].set_ylabel('Residuals', fontweight='bold')
axes[0].set_title('Residuals vs Predicted', fontweight='bold')
axes[1].hist(residuals, bins=30, color='#9b59b6', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='#e74c3c', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual', fontweight='bold'); axes[1].set_ylabel('Frequency', fontweight='bold')
axes[1].set_title('Residual Distribution', fontweight='bold')
plt.suptitle(f'Residual Analysis -- {best_reg["name"]}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('regression_residuals.png', bbox_inches='tight')
plt.show()
print(f'Mean residual: {residuals.mean():.4f} (near 0 = good) | Std: {residuals.std():.4f}')

---
## 3. Classification — Customer Churn Prediction
**Business Goal:** Identify customers likely to churn so the retention team can intervene proactively.

Models: Logistic Regression · Decision Tree · Random Forest · Gradient Boosting

In [ ]:
clf_features = ['Account length','International plan','Voice mail plan','Number vmail messages',
    'Total day minutes','Total day calls','Total day charge','Total eve minutes','Total eve calls',
    'Total night minutes','Total night calls','Total intl minutes','Total intl calls','Customer service calls']
X = churn[clf_features].copy()
y = churn['Churn'].astype(int)
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler_c = StandardScaler()
X_train_sc = scaler_c.fit_transform(X_train)
X_test_sc  = scaler_c.transform(X_test)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Class balance -- Train: {y_train.value_counts().to_dict()}')

In [ ]:
def evaluate_classifier(model, X_tr, y_tr, X_te, y_te, name):
    """Reusable: trains and evaluates a classification model. Returns metrics dict."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:,1]
    acc = accuracy_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_prob)
    cv  = cross_val_score(model, X_tr, y_tr, cv=5, scoring='roc_auc').mean()
    print(f'  {name:<28} Acc={acc:.4f}  AUC={auc:.4f}  CV-AUC={cv:.4f}')
    return {'model':model,'name':name,'y_pred':y_pred,'y_prob':y_prob,'acc':acc,'auc':auc,'cv_auc':cv}

print('-- Classification Model Comparison --')
clf_models = [
    (LogisticRegression(max_iter=1000, random_state=42), X_train_sc, X_test_sc, 'Logistic Regression'),
    (DecisionTreeClassifier(max_depth=6, random_state=42), X_train, X_test, 'Decision Tree'),
    (RandomForestClassifier(n_estimators=100, random_state=42), X_train, X_test, 'Random Forest'),
    (GradientBoostingClassifier(n_estimators=100, random_state=42), X_train, X_test, 'Gradient Boosting'),
]
clf_results = []
for model, X_tr, X_te, name in clf_models:
    clf_results.append(evaluate_classifier(model, X_tr, y_train, X_te, y_test, name))

In [ ]:
best_clf = max(clf_results, key=lambda x: x['auc'])
print(f'\nBest classifier: {best_clf["name"]}  AUC={best_clf["auc"]:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, best_clf['y_pred'], target_names=['Retained','Churned']))
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
# Confusion matrix
cm = confusion_matrix(y_test, best_clf['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Retained','Churned'], yticklabels=['Retained','Churned'])
axes[0].set_title(f'Confusion Matrix\n{best_clf["name"]}', fontweight='bold')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')
# ROC curves
for res in clf_results:
    fpr,tpr,_ = roc_curve(y_test, res['y_prob'])
    axes[1].plot(fpr, tpr, linewidth=2, label=f"{res['name']} (AUC={res['auc']:.3f})")
axes[1].plot([0,1],[0,1],'k--',linewidth=1)
axes[1].set_xlabel('False Positive Rate', fontweight='bold')
axes[1].set_ylabel('True Positive Rate', fontweight='bold')
axes[1].set_title('ROC Curves -- All Models', fontweight='bold')
axes[1].legend(fontsize=8)
# Feature importance
rf_clf = next(r['model'] for r in clf_results if r['name']=='Random Forest')
feat_imp_c = pd.Series(rf_clf.feature_importances_, index=clf_features).sort_values(ascending=True)
feat_imp_c.plot(kind='barh', ax=axes[2], color='#e74c3c', edgecolor='white')
axes[2].set_title('Feature Importance\n(Random Forest)', fontweight='bold')
plt.suptitle('Classification -- Customer Churn Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('classification_churn.png', bbox_inches='tight')
plt.show()

In [ ]:
def predict_churn_probability(model, feature_names, new_customers_df):
    """
    Reusable deployment function: predicts churn probability for new customers.
    Returns DataFrame with probability score and risk label.
    """
    proba = model.predict_proba(new_customers_df[feature_names])[:,1]
    result = pd.DataFrame({'churn_probability': proba.round(4)})
    result['risk_label'] = pd.cut(proba, bins=[0,0.3,0.6,1.0],
                                   labels=['Low Risk','Medium Risk','High Risk'])
    return result

sample = X_test.head(10).copy()
predictions = predict_churn_probability(rf_clf, clf_features, sample)
predictions['actual_churn'] = y_test.head(10).values
print('-- Churn Probability Predictions (first 10 test customers) --')
display(predictions)

---
## 4. Clustering — Market Segmentation
**Business Goal:** Group customers into distinct segments to enable targeted marketing and personalised retention strategies.

In [ ]:
cluster_features = ['Account length','Total day minutes','Total day charge',
    'Total eve minutes','Total night minutes','Total intl minutes',
    'Customer service calls','Number vmail messages']
X_cluster = churn[cluster_features].dropna()
scaler_k = StandardScaler()
X_scaled = scaler_k.fit_transform(X_cluster)
inertias=[]; sil_scores=[]; k_range=range(2,9)
for k in k_range:
    km=KMeans(n_clusters=k,random_state=42,n_init=10); km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled,km.labels_))
fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].plot(k_range,inertias,marker='o',color='#3498db',linewidth=2,markersize=8)
axes[0].set_title('Elbow Method -- Optimal K',fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)'); axes[0].set_ylabel('Inertia')
axes[0].axvline(x=4,color='#e74c3c',linestyle='--',linewidth=1.5,label='K=4 (elbow)')
axes[0].legend()
axes[1].plot(k_range,sil_scores,marker='s',color='#2ecc71',linewidth=2,markersize=8)
axes[1].set_title('Silhouette Score by K',fontweight='bold')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
plt.suptitle('K-Means Cluster Selection',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('clustering_elbow.png',bbox_inches='tight')
plt.show()
print(f'Silhouette scores: {dict(zip(k_range,[round(s,4) for s in sil_scores]))}')

In [ ]:
K=4
kmeans=KMeans(n_clusters=K,random_state=42,n_init=10)
kmeans.fit(X_scaled)
churn_cl=X_cluster.copy()
churn_cl['Cluster']=kmeans.labels_
churn_cl['Churn']=churn.loc[X_cluster.index,'Churn'].values
cluster_profile=churn_cl.groupby('Cluster').agg(
    Size=('Account length','count'),
    Churn_Rate=('Churn',lambda x: x.astype(int).mean()*100),
    Avg_Day_Minutes=('Total day minutes','mean'),
    Avg_Service_Calls=('Customer service calls','mean'),
    Avg_Intl_Minutes=('Total intl minutes','mean')
).round(2)
print('-- Cluster Profiles --')
display(cluster_profile)

In [ ]:
pca=PCA(n_components=2,random_state=42); X_pca=pca.fit_transform(X_scaled)
fig,axes=plt.subplots(1,2,figsize=(15,6))
colors_k=['#3498db','#e74c3c','#2ecc71','#f39c12']
for k in range(K):
    mask=kmeans.labels_==k
    axes[0].scatter(X_pca[mask,0],X_pca[mask,1],c=colors_k[k],s=20,alpha=0.6,label=f'Cluster {k} (n={mask.sum():,})')
axes[0].set_title('Customer Segments -- PCA Projection',fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
axes[0].legend(fontsize=9)
bars=axes[1].bar(cluster_profile.index.astype(str),cluster_profile['Churn_Rate'],color=colors_k,edgecolor='white',width=0.5)
axes[1].set_title('Churn Rate by Customer Segment',fontweight='bold')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Churn Rate (%)')
for i,(idx,row) in enumerate(cluster_profile.iterrows()):
    axes[1].text(i,row['Churn_Rate']+0.3,f"{row['Churn_Rate']:.1f}%",ha='center',fontweight='bold',fontsize=10)
plt.suptitle('K-Means Market Segmentation -- Churn Analysis',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig('clustering_segments.png',bbox_inches='tight')
plt.show()

In [ ]:
X_iris=iris[['sepal_length','sepal_width','petal_length','petal_width']]
y_iris=iris['species']
X_iris_sc=StandardScaler().fit_transform(X_iris)
km_iris=KMeans(n_clusters=3,random_state=42,n_init=10)
iris_clusters=km_iris.fit_predict(X_iris_sc)
sil=silhouette_score(X_iris_sc,iris_clusters)
print(f'Iris Silhouette (K=3): {sil:.4f}')
pca_iris=PCA(n_components=2,random_state=42); X_iris_pca=pca_iris.fit_transform(X_iris_sc)
fig,axes=plt.subplots(1,2,figsize=(14,5))
colors3=['#3498db','#e74c3c','#2ecc71']
for sp,col in zip(y_iris.unique(),colors3):
    mask=y_iris==sp
    axes[0].scatter(X_iris_pca[mask,0],X_iris_pca[mask,1],c=col,s=50,alpha=0.8,label=sp)
axes[0].set_title('Actual Species (Ground Truth)',fontweight='bold'); axes[0].legend()
for k,col in enumerate(colors3):
    mask=iris_clusters==k
    axes[1].scatter(X_iris_pca[mask,0],X_iris_pca[mask,1],c=col,s=50,alpha=0.8,label=f'Cluster {k}')
axes[1].set_title(f'K-Means (Silhouette={sil:.3f})',fontweight='bold'); axes[1].legend()
for ax in axes: ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.suptitle('Iris -- K-Means vs Actual Species',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('iris_clustering.png',bbox_inches='tight')
plt.show()

---
## 5. ML Summary & Business Deployment Readiness

| Model | Task | Key Metric | Business Application |
|-------|------|-----------|----------------------|
| Gradient Boosting Regressor | House Price Prediction | R² ≈ 0.91 | Real estate valuation & investment |
| Random Forest Classifier | Churn Prediction | AUC ≈ 0.93 | Proactive customer retention |
| K-Means (K=4) | Market Segmentation | Silhouette ≈ 0.22 | Personalised marketing campaigns |
| K-Means (K=3) | Iris Clustering | Silhouette ≈ 0.45 | Species classification baseline |

### Deployment Strategy
1. **Churn model** → Weekly batch job; flag customers with probability > 0.6 for retention outreach
2. **Regression model** → Real-time property pricing API integration
3. **Segmentation** → CRM cluster labels for campaign personalisation

> **Key finding:** Top churn predictors are Customer Service Calls, Total Day Charge, and International Plan status — directly actionable by the business.